패키지

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
from scipy.stats import norm, shapiro, kstest, probplot, jarque_bera
import plotly
import plotly.express as px
import nbformat

from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import gc

from collections import Counter

In [3]:
# 모델링 관련 패키지
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import xgboost as xgb
from xgboost.callback import EarlyStopping

import optuna

from sklearn.metrics import mean_squared_error

c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


한글 패치

In [4]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 나눔고딕 경로 (Windows 설치 위치)
font_path = "C:/Windows/Fonts/NanumGothic.ttf"
fontprop = fm.FontProperties(fname=font_path)

plt.rc('font', family='Malgun Gothic')   # 기본 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

인접행렬 불러오기

In [7]:
# csv 불러와서 행렬로 만들기
adj_fatigue_df = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\연구기록\simple_adj.csv", encoding="utf-8-sig", index_col=0)

adj_fatigue = adj_fatigue_df.values
stations = adj_fatigue_df.index.tolist()

station_to_num = {station: i for i, station in enumerate(stations)}
num_to_station = {i: station for station, i in station_to_num.items()}

adj_fatigue[station_to_num['구산']][station_to_num['응암']]

np.float64(0.259831352)

In [8]:
# csv 불러와서 행렬로 만들기
adj_dist_df = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\코드\경로추천 알고리즘\distance_adj.csv", encoding="utf-8-sig", index_col=0)

adj_dist = adj_dist_df.values
stations = adj_dist_df.index.tolist()

station_to_num = {station: i for i, station in enumerate(stations)}
num_to_station = {i: station for station, i in station_to_num.items()}

adj_dist[station_to_num['구산']][station_to_num['응암']]

np.float64(0.6519537729620372)

가중치에 따라 합쳐서 거리 + 혼잡도 인접행렬 만들기

In [11]:
# 혼잡도에 부여할 가중치 하이퍼파라미터
beta = 0.5

adj = (1-beta) * adj_dist + beta * adj_fatigue

In [12]:
len(stations)

238

In [13]:
INF = 100000  # 무한대 대신 충분히 큰 값

# 지하철 노선도
VERTEX_COUNT = len(stations)

# 다익스트라 알고리즘

In [18]:
def dijkstra(start_v, end_v, beta):
    adj = (1-beta) * adj_dist + beta * adj_fatigue


    visited = [False] * VERTEX_COUNT
    distance = [INF] * VERTEX_COUNT
    parent = [-1] * VERTEX_COUNT  # parent[i] = i로 오기 직전 정점

    distance[start_v] = 0

    for _ in range(VERTEX_COUNT):
        # 아직 방문하지 않은 정점 중 최단거리 정점 선택
        min_dist = INF
        current_v = -1

        for i in range(VERTEX_COUNT):
            if not visited[i] and distance[i] < min_dist:
                min_dist = distance[i]
                current_v = i

        # 더 이상 갈 수 있는 정점이 없으면 종료
        if current_v == -1:
            break

        visited[current_v] = True

        # end_v를 찾았으면 조기 종료 가능
        if current_v == end_v:
            break

        # current_v를 거쳐 가는 경로가 더 짧으면 갱신
        for next_v in range(VERTEX_COUNT):
            if not visited[next_v] and adj[current_v][next_v] != INF:
                new_dist = distance[current_v] + adj[current_v][next_v]
                if new_dist < distance[next_v]:
                    distance[next_v] = new_dist
                    parent[next_v] = current_v

    # 도착점에 도달 불가능한 경우
    if distance[end_v] == INF:
        print("도달할 수 없는 경로입니다.")
        return

    # 경로 복원
    route = []
    now = end_v
    while now != -1:
        route.append(now)
        now = parent[now]
    route.reverse()

    # 출력
    for i, v in enumerate(route):
        if i == 0:
            print(f"{num_to_station[v]} 시작")
        elif i == len(route) - 1:
            print(f"{num_to_station[v]} 도착")
        else:
            print(f"{num_to_station[v]} 방문")

    print(f"\n최단 거리: {distance[end_v]}")

# 서울역 - 한양대에 적용

In [43]:
start = '서울역'
final = '한양대'

print(dijkstra(station_to_num[start], station_to_num[final], beta = 0))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
신당 방문
상왕십리 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 3.189455798887285
None


In [46]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.1))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
신당 방문
상왕십리 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 3.076084969698557
None


In [47]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.2))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
신당 방문
상왕십리 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.962714140509828
None


In [48]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.3))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.8448041753014888
None


In [49]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.4))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.6350932374012768
None


In [44]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.5))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.4253822995010634
None


In [50]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.6))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.2156713616008505
None


In [51]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.7))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 2.0059604237006385
None


In [52]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.8))

서울역 시작
회현.남대문시장. 방문
명동 방문
충무로 방문
동대문역사문화공원.DDP. 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 1.7962494858004254
None


In [69]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.87))

서울역 시작
숙대입구.갈월. 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 1.6468408841307383
None


In [57]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.88))

서울역 시작
숙대입구.갈월. 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신금호 방문
행당 방문
왕십리.성동구청. 방문
한양대 도착

최단 거리: 1.6083899526591428
None


In [60]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.89))

서울역 시작
숙대입구.갈월. 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신당 방문
동묘앞 방문
신설동 방문
용두.동대문구청. 방문
신답 방문
용답 방문
성수 방문
뚝섬 방문
한양대 도착

최단 거리: 1.5545882470193733
None


In [53]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.9))

서울역 시작
숙대입구.갈월. 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신당 방문
동묘앞 방문
신설동 방문
용두.동대문구청. 방문
신답 방문
용답 방문
성수 방문
뚝섬 방문
한양대 도착

최단 거리: 1.4872856507448846
None


In [54]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 1))

서울역 시작
숙대입구.갈월. 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신당 방문
동묘앞 방문
신설동 방문
용두.동대문구청. 방문
신답 방문
용답 방문
성수 방문
뚝섬 방문
한양대 도착

최단 거리: 0.8142596879999999
None


In [72]:
start = '광명사거리'
final = '잠실.송파구청.'

print(dijkstra(station_to_num[start], station_to_num[final], beta = 0))

광명사거리 시작
철산 방문
가산디지털단지 방문
남구로 방문
대림.구로구청. 방문
구로디지털단지 방문
신대방 방문
신림 방문
봉천 방문
서울대입구.관악구청. 방문
낙성대.강감찬. 방문
사당 방문
방배 방문
서초 방문
교대.법원.검찰청. 방문
강남 방문
역삼 방문
선릉 방문
삼성.무역센터. 방문
종합운동장 방문
잠실새내 방문
잠실.송파구청. 도착

최단 거리: 11.130231781001156
None


In [74]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 0.5))

광명사거리 시작
철산 방문
가산디지털단지 방문
남구로 방문
대림.구로구청. 방문
신풍 방문
보라매 방문
신대방삼거리 방문
장승배기 방문
상도 방문
숭실대입구.살피재. 방문
남성 방문
이수 방문
내방 방문
고속터미널 방문
교대.법원.검찰청. 방문
강남 방문
역삼 방문
선릉 방문
삼성.무역센터. 방문
종합운동장 방문
잠실새내 방문
잠실.송파구청. 도착

최단 거리: 11.054616824799405
None


In [73]:
print(dijkstra(station_to_num[start], station_to_num[final], beta = 1))

광명사거리 시작
철산 방문
가산디지털단지 방문
남구로 방문
대림.구로구청. 방문
신도림 방문
문래 방문
영등포구청 방문
당산 방문
합정 방문
상수 방문
광흥창.서강. 방문
대흥.서강대앞. 방문
공덕 방문
효창공원앞 방문
삼각지 방문
녹사평.용산구청. 방문
이태원 방문
한강진 방문
버티고개 방문
약수 방문
청구 방문
신당 방문
동묘앞 방문
신설동 방문
용두.동대문구청. 방문
신답 방문
용답 방문
성수 방문
건대입구 방문
구의.광진구청. 방문
강변.동서울터미널. 방문
잠실나루 방문
잠실.송파구청. 도착

최단 거리: 8.383387047000001
None
